# Training Metrics and Hardware Utilization Comparison (Baseline vs. Optimized)

This notebook loads the advanced metrics from `metrics_advanced.json` and the baseline metrics from `metrics.json` to analyze and compare the performance, throughput, resource consumption, and GPU efficiency (TFLOPs, MFU, bandwidth) of the baseline vs. optimized transformer models.

In [1]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
# Load the metrics data robustly
import os
import json
import pandas as pd

# Load Baseline (Chapter 1)
metrics_file = "metrics.json"
if not os.path.exists(metrics_file):
    metrics_file = "../../chapter_1_transformers/basic_transformer_implementation/metrics.json"
if not os.path.exists(metrics_file):
    metrics_file = os.path.join("chapter_1_transformers", "basic_transformer_implementation", "metrics.json")
if not os.path.exists(metrics_file):
    metrics_file = "/root/inference-engineering-guide/chapter_1_transformers/basic_transformer_implementation/metrics.json"

with open(metrics_file, "r") as f:
    data = json.load(f)
df = pd.DataFrame(data)

# Load Optimized (Chapter 2)
metrics_adv_file = "metrics_advanced.json"
if not os.path.exists(metrics_adv_file):
    metrics_adv_file = os.path.join("chapter_2_hardware_optimization", "advanced_transformer_implementation", "metrics_advanced.json")
if not os.path.exists(metrics_adv_file):
    metrics_adv_file = "/root/inference-engineering-guide/chapter_2_hardware_optimization/advanced_transformer_implementation/metrics_advanced.json"

with open(metrics_adv_file, "r") as f:
    data_adv = json.load(f)
df_adv = pd.DataFrame(data_adv)

print("Baseline loaded shape:", df.shape)
print("Optimized loaded shape:", df_adv.shape)


Baseline loaded shape: (10, 13)
Optimized loaded shape: (3, 13)


## 1. Machine Learning Metrics (Loss, Perplexity, Accuracy)

In [3]:
# Plot Train Loss and Dev Accuracy (Comparing Baseline and Optimized)
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Baseline
fig.add_trace(
    go.Scatter(x=df['epoch'], y=df['train_loss'], name="Baseline Train Loss", mode="lines+markers", line=dict(color="royalblue", width=2)),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=df['epoch'], y=df['val_acc'] * 100, name="Baseline Val Accuracy", mode="lines+markers", line=dict(color="forestgreen", width=2)),
    secondary_y=True,
)

# Optimized
fig.add_trace(
    go.Scatter(x=df_adv['epoch'], y=df_adv['train_loss'], name="Optimized Train Loss", mode="lines+markers", line=dict(color="cyan", width=3)),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=df_adv['epoch'], y=df_adv['val_acc'] * 100, name="Optimized Val Accuracy", mode="lines+markers", line=dict(color="limegreen", width=3)),
    secondary_y=True,
)

fig.update_layout(
    title_text="Training Loss and Validation Accuracy Comparison (Baseline vs Optimized)",
    xaxis_title="Epoch",
    template="plotly_dark",
    legend=dict(x=0.01, y=0.99)
)

fig.update_yaxes(title_text="Cross Entropy Loss", secondary_y=False)
fig.update_yaxes(title_text="Accuracy (%)", secondary_y=True)

fig.show()


## 2. Hardware Utilization (Peak VRAM Allocated vs. Reserved)

VRAM allocation tracking is crucial in LLM engineering. 
- **Allocated Memory**: The VRAM actively holding tensors.
- **Reserved Memory**: The VRAM cached by PyTorch's memory allocator (caching allocator) to avoid the high overhead of repeatedly querying the CUDA driver.

In [4]:
fig_vram = go.Figure()
# Baseline
fig_vram.add_trace(go.Bar(
    x=[f"Base E{e}" for e in df['epoch']], 
    y=df['peak_vram_allocated_mb'],
    name='Baseline Allocated (MB)',
    marker_color='royalblue'
))
fig_vram.add_trace(go.Bar(
    x=[f"Base E{e}" for e in df['epoch']], 
    y=df['peak_vram_reserved_mb'],
    name='Baseline Reserved (MB)',
    marker_color='cornflowerblue'
))

# Optimized
fig_vram.add_trace(go.Bar(
    x=[f"Opt E{e}" for e in df_adv['epoch']], 
    y=df_adv['peak_vram_allocated_mb'],
    name='Optimized Allocated (MB)',
    marker_color='crimson'
))
fig_vram.add_trace(go.Bar(
    x=[f"Opt E{e}" for e in df_adv['epoch']], 
    y=df_adv['peak_vram_reserved_mb'],
    name='Optimized Reserved (MB)',
    marker_color='lightcoral'
))

fig_vram.update_layout(
    barmode='group',
    title_text='Peak GPU VRAM Usage Comparison (Baseline vs. Optimized)',
    xaxis_title='Epoch / Configuration',
    yaxis_title='VRAM (MB)',
    template='plotly_dark',
    legend=dict(x=0.01, y=0.99)
)
fig_vram.show()


## 3. Data Processing Rate (Throughput vs. Goodput)

- **Throughput**: Total raw tokens (characters) processed per second.
- **Goodput**: Useful tokens processed per second (excluding padding).

### Why do the Throughput and Goodput lines overlap exactly?
In this character-level implementation, we slice the text corpus into fixed chunks of 20 characters directly. **No padding tokens (`<pad>`) are used.** Since $100\%$ of the processed tokens are useful learning tokens, **Goodput is mathematically identical to Throughput**, resulting in the two lines overlapping perfectly on the chart.

In [5]:
fig_tp = go.Figure()
fig_tp.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['throughput_tokens_sec'],
    name='Baseline Throughput',
    mode='lines+markers',
    line=dict(color='darkorange', width=2)
))
fig_tp.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['goodput_tokens_sec'],
    name='Baseline Goodput',
    mode='lines+markers',
    line=dict(color='orange', width=1, dash='dash')
))

fig_tp.add_trace(go.Scatter(
    x=df_adv['epoch'], 
    y=df_adv['throughput_tokens_sec'],
    name='Optimized Throughput',
    mode='lines+markers',
    line=dict(color='cyan', width=3)
))
fig_tp.add_trace(go.Scatter(
    x=df_adv['epoch'], 
    y=df_adv['goodput_tokens_sec'],
    name='Optimized Goodput',
    mode='lines+markers',
    line=dict(color='deepskyblue', width=2, dash='dash')
))

fig_tp.update_layout(
    title_text='Data Processing Rate (Throughput vs. Goodput) Comparison',
    xaxis_title='Epoch',
    yaxis_title='Tokens / Second',
    template='plotly_dark',
    legend=dict(x=0.01, y=0.99)
)
fig_tp.show()


## 4. Compute Performance (TFLOPs and Model FLOPs Utilization)

- **Achieved TFLOPs/sec**: The absolute processing rate of raw floating point operations.
- **Model FLOPs Utilization (MFU %)**: The ratio of achieved compute performance to the hardware's peak theoretical performance. For your RTX 4070 Super, peak dense FP16 tensor core throughput is **142.2 TFLOPs/sec**.

In [6]:
# Plot Achieved TFLOPs/sec and MFU comparing baseline vs optimized
fig_tflops = go.Figure()
fig_tflops.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['tflops_per_sec'], 
    name="Baseline TFLOPs/sec", 
    mode="lines+markers", 
    line=dict(color="mediumpurple", width=2)
))
fig_tflops.add_trace(go.Scatter(
    x=df_adv['epoch'], 
    y=df_adv['tflops_per_sec'], 
    name="Optimized TFLOPs/sec", 
    mode="lines+markers", 
    line=dict(color="orchid", width=3)
))
fig_tflops.update_layout(
    title_text="Achieved Compute Performance Comparison (TFLOPs/sec)",
    xaxis_title="Epoch",
    yaxis_title="TFLOPs / Second",
    template="plotly_dark",
    legend=dict(x=0.01, y=0.99)
)
fig_tflops.show()

# Plot MFU (%)
fig_mfu = go.Figure()
fig_mfu.add_trace(go.Scatter(
    x=df['epoch'], 
    y=df['mfu_percent'], 
    name="Baseline MFU (%)", 
    mode="lines+markers", 
    line=dict(color="hotpink", width=2)
))
fig_mfu.add_trace(go.Scatter(
    x=df_adv['epoch'], 
    y=df_adv['mfu_percent'], 
    name="Optimized MFU (%)", 
    mode="lines+markers", 
    line=dict(color="deeppink", width=3)
))
fig_mfu.update_layout(
    title_text="Model FLOPs Utilization Comparison (MFU %)",
    xaxis_title="Epoch",
    yaxis_title="MFU (%)",
    template="plotly_dark",
    legend=dict(x=0.01, y=0.99)
)
fig_mfu.show()


## 5. Roofline Model Analysis

A Roofline chart maps achieved performance (TFLOPs/sec) on the y-axis against the arithmetic intensity (FLOPs/Byte) on the x-axis. This highlights whether your training workload is memory-bandwidth bound or compute bound.

In [7]:
# Plot Roofline chart overlaying Baseline and Optimized runs
import numpy as np

# 1. Hardware Limits for RTX 4070 Super
peak_bandwidth_gb_s = 504.0
peak_fp16_tensor_tflops = 142.2
peak_fp32_vector_tflops = 35.5

# 2. Generate Roofline boundary lines
intensities = np.logspace(-3, 3, 500)
perf_fp16_ceiling = np.minimum(intensities * (peak_bandwidth_gb_s / 1000.0), peak_fp16_tensor_tflops)
perf_fp32_ceiling = np.minimum(intensities * (peak_bandwidth_gb_s / 1000.0), peak_fp32_vector_tflops)

# 3. Create the figure
fig_roofline = go.Figure()

fig_roofline.add_trace(go.Scatter(
    x=intensities,
    y=perf_fp16_ceiling,
    mode='lines',
    name='FP16 Tensor Core Peak (142.2 TFLOPs)',
    line=dict(color='crimson', width=3, dash='dash')
))

fig_roofline.add_trace(go.Scatter(
    x=intensities,
    y=perf_fp32_ceiling,
    mode='lines',
    name='FP32 Vector Peak (35.5 TFLOPs)',
    line=dict(color='orange', width=2, dash='dot')
))

# 4. Extract data points from baseline training metrics
epochs = df['epoch'].tolist()
tflops_per_epoch = df['tflops_per_sec'].tolist()
intensities_per_epoch = df['arithmetic_intensity'].tolist()

fig_roofline.add_trace(go.Scatter(
    x=intensities_per_epoch,
    y=tflops_per_epoch,
    mode='markers+text',
    name='Baseline (412K params, FP32, Uncompiled)',
    text=[f"Base E{e}" for e in epochs],
    textposition="bottom center",
    marker=dict(color='cyan', size=10, symbol='circle', line=dict(color='white', width=1))
))

# 5. Extract data points from optimized training metrics
epochs_adv = df_adv['epoch'].tolist()
tflops_per_epoch_adv = df_adv['tflops_per_sec'].tolist()
intensities_per_epoch_adv = df_adv['arithmetic_intensity'].tolist()

fig_roofline.add_trace(go.Scatter(
    x=intensities_per_epoch_adv,
    y=tflops_per_epoch_adv,
    mode='markers+text',
    name='Optimized (19.5M params, FP16 AMP, compiled)',
    text=[f"Opt E{e}" for e in epochs_adv],
    textposition="top left",
    marker=dict(color='gold', size=14, symbol='star', line=dict(color='white', width=1))
))

# 6. Format layout (log-log scale)
fig_roofline.update_layout(
    title=dict(text="NVIDIA GeForce RTX 4070 Super Roofline Analysis", font=dict(size=18)),
    xaxis=dict(
        title="Arithmetic Intensity (FLOPs / Byte)",
        type="log",
        gridcolor="#283442",
        zerolinecolor="#283442"
    ),
    yaxis=dict(
        title="Performance (TFLOPs / sec)",
        type="log",
        gridcolor="#283442",
        zerolinecolor="#283442",
        range=[-3, 2.5]
    ),
    template="plotly_dark",
    legend=dict(x=0.02, y=0.98),
    height=600
)
fig_roofline.show()
